# Fourier Series of a Square Wave — Interactive Lab

**Signals & Systems / Electronics Engineering**

In this lab you will decompose a periodic square wave into a sum of sinusoids (its **Fourier series**), inspect its **amplitude** and **phase spectra**, and rebuild the signal from a limited number of harmonics.

### Learning objectives
By the end of this notebook you should be able to:
1. Explain why a square wave contains only odd harmonics (for 50 % duty cycle) and why their amplitudes fall as $1/n$.
2. Read and interpret one‑sided amplitude and phase spectra.
3. Observe how the reconstruction improves as harmonics are added, and recognise the **Gibbs phenomenon**.
4. Predict how a **time shift** affects the phase spectrum but not the amplitude spectrum.
5. Predict how changing the **duty cycle** changes the harmonic content.

> **How to use:** run all cells (*Run → Run All* or `Shift+Enter` cell by cell). The interactive panel is in Section 3.

## 1. Theory recap

A periodic signal $x(t)$ with period $T_0$ and fundamental frequency $f_0 = 1/T_0$ (angular $\omega_0 = 2\pi f_0$) can be written as

$$
x(t) = \sum_{n=-\infty}^{+\infty} c_n \, e^{j n \omega_0 t},
\qquad
c_n = \frac{1}{T_0}\int_{0}^{T_0} x(t)\, e^{-j n \omega_0 t}\, dt .
$$

For a real signal $c_{-n} = c_n^*$, so we can use the equivalent **one‑sided (amplitude–phase) form**

$$
x(t) = A_0 + \sum_{n=1}^{\infty} A_n \cos\!\left(n\omega_0 t + \varphi_n\right),
\qquad A_0 = c_0,\quad A_n = 2|c_n|,\quad \varphi_n = \angle c_n .
$$

The plots of $A_n$ and $\varphi_n$ versus frequency $n f_0$ are the **amplitude spectrum** and **phase spectrum**.

### The square wave used here
A bipolar pulse train of amplitude $A$, duty cycle $D$ (fraction of the period at $+A$), delayed by $\tau$:

$$
x(t) = \begin{cases} +A, & 0 \le (t-\tau) \bmod T_0 < D\,T_0 \\ -A, & \text{otherwise} \end{cases}
$$

Its coefficients (derive these yourself as an exercise!) are

$$
c_0 = A(2D-1), \qquad
c_n = \frac{A\left(1 - e^{-j2\pi n D}\right)}{j\pi n}\; e^{-j n \omega_0 \tau}, \quad n \neq 0 .
$$

**Classic case** ($D = 0.5$, $\tau = 0$): $c_n = \dfrac{2A}{j\pi n}$ for odd $n$ and $0$ for even $n$, hence

$$
x(t) = \frac{4A}{\pi}\left[\sin(\omega_0 t) + \tfrac{1}{3}\sin(3\omega_0 t) + \tfrac{1}{5}\sin(5\omega_0 t) + \dots\right],
$$

i.e. $A_n = \dfrac{4A}{n\pi}$ and $\varphi_n = -90^\circ$ (because $\sin\theta = \cos(\theta - 90^\circ)$).

## 2. Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from ipywidgets import interact
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found: install it with  pip install ipywidgets  "
          "(the plots below still work, just without sliders).")

# If you are in Google Colab, uncomment the next two lines:
# from google.colab import output
# output.enable_custom_widget_manager()

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

In [2]:
def square_wave(t, A=1.0, f0=1.0, D=0.5, tau=0.0):
    '''Ideal bipolar square wave: +A for a fraction D of each period, -A otherwise.'''
    T0 = 1.0 / f0
    phase = np.mod(t - tau, T0) / T0          # position inside the period, 0..1
    return np.where(phase < D, A, -A)


def fourier_coefficients(N, A=1.0, f0=1.0, D=0.5, tau=0.0):
    '''Complex coefficients c_0 ... c_N of the square wave (analytic expressions).'''
    n = np.arange(N + 1)
    c = np.zeros(N + 1, dtype=complex)
    c[0] = A * (2 * D - 1)
    k = n[1:]
    c[1:] = A * (1 - np.exp(-1j * 2 * np.pi * k * D)) / (1j * np.pi * k)
    c[1:] *= np.exp(-1j * 2 * np.pi * k * f0 * tau)  # time shift -> linear phase
    return n, c


def one_sided_spectrum(c, tol=1e-9):
    '''Amplitudes A_n and phases phi_n (degrees). Phase is undefined (NaN) where A_n = 0.'''
    amp = np.abs(c).copy()
    amp[1:] *= 2
    phase = np.degrees(np.angle(c))
    phase[amp < tol] = np.nan
    return amp, phase


def reconstruct(t, c, f0=1.0):
    '''Partial Fourier sum  A_0 + sum_{n=1}^{N} A_n cos(n w0 t + phi_n).'''
    x = np.real(c[0]) * np.ones_like(t)
    for n in range(1, len(c)):
        x += 2 * np.abs(c[n]) * np.cos(2 * np.pi * n * f0 * t + np.angle(c[n]))
    return x

# Quick sanity check against the textbook result 4A/(n*pi) for odd n
n, c = fourier_coefficients(7)
amp, ph = one_sided_spectrum(c)
for k in range(1, 8):
    expected = 4 / (k * np.pi) if k % 2 else 0.0
    print(f"n={k}:  A_n = {amp[k]:.4f}   (expected {expected:.4f})   phase = {ph[k]:7.1f} deg")

n=1:  A_n = 1.2732   (expected 1.2732)   phase =   -90.0 deg
n=2:  A_n = 0.0000   (expected 0.0000)   phase =     nan deg
n=3:  A_n = 0.4244   (expected 0.4244)   phase =   -90.0 deg
n=4:  A_n = 0.0000   (expected 0.0000)   phase =     nan deg
n=5:  A_n = 0.2546   (expected 0.2546)   phase =   -90.0 deg
n=6:  A_n = 0.0000   (expected 0.0000)   phase =     nan deg
n=7:  A_n = 0.1819   (expected 0.1819)   phase =   -90.0 deg


In [3]:
def plot_decomposition(N=5, D=0.5, tau=0.0, show_harmonics=False, A=1.0, f0=1.0, periods=2):
    T0 = 1 / f0
    t = np.linspace(-0.25 * T0, (periods - 0.25) * T0, 4000)
    x = square_wave(t, A, f0, D, tau)

    n, c = fourier_coefficients(N, A, f0, D, tau)
    amp, phase = one_sided_spectrum(c)
    x_hat = reconstruct(t, c, f0)

    # Figures of merit
    err = x - x_hat
    mse = np.mean(err ** 2)
    P_total = A ** 2                                   # power of a +/-A square wave
    P_kept = amp[0] ** 2 + 0.5 * np.sum(amp[1:] ** 2)  # Parseval
    overshoot = (x_hat.max() - A) / (2 * A) * 100      # relative to the jump 2A

    fig = plt.figure(figsize=(13, 9), constrained_layout=True)
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.1])

    # Original
    ax0 = fig.add_subplot(gs[0, :])
    ax0.plot(t, x, color="black", lw=2)
    ax0.set_title("Original signal  x(t)")
    ax0.set_ylabel("Amplitude"); ax0.set_ylim(-1.5 * A, 1.5 * A)

    # Amplitude spectrum
    f = n * f0
    ax1 = fig.add_subplot(gs[1, 0])
    ml, sl, bl = ax1.stem(f, amp, basefmt=" ")
    plt.setp(ml, color="tab:blue"); plt.setp(sl, color="tab:blue")
    ax1.set_title(f"Amplitude spectrum  $A_n$  (N = {N})")
    ax1.set_xlabel("Frequency (multiples of $f_0$)"); ax1.set_ylabel("$A_n$")
    ax1.set_xlim(-0.5, max(N, 5) + 0.5); ax1.set_ylim(0, 1.4 * A)
    if N <= 25:
        ax1.set_xticks(f)

    # Phase spectrum
    ax2 = fig.add_subplot(gs[1, 1])
    valid = ~np.isnan(phase)
    ml, sl, bl = ax2.stem(f[valid], phase[valid], basefmt=" ")
    plt.setp(ml, color="tab:orange"); plt.setp(sl, color="tab:orange")
    ax2.axhline(0, color="gray", lw=0.8)
    ax2.set_title(r"Phase spectrum  $\varphi_n$  (only where $A_n \neq 0$)")
    ax2.set_xlabel("Frequency (multiples of $f_0$)"); ax2.set_ylabel("Phase (deg)")
    ax2.set_ylim(-200, 200); ax2.set_yticks([-180, -90, 0, 90, 180])
    ax2.set_xlim(-0.5, max(N, 5) + 0.5)
    if N <= 25:
        ax2.set_xticks(f)

    # Reconstruction
    ax3 = fig.add_subplot(gs[2, :])
    if show_harmonics:
        for k in range(1, N + 1):
            if amp[k] > 1e-9:
                ax3.plot(t, amp[k] * np.cos(2 * np.pi * k * f0 * t + np.angle(c[k])),
                         lw=0.8, alpha=0.6, label=f"n={k}" if k <= 9 else None)
    ax3.plot(t, x, color="black", lw=1, ls="--", alpha=0.6, label="original")
    ax3.plot(t, x_hat, color="tab:red", lw=2, label="reconstruction")
    ax3.set_title(f"Reconstructed signal with harmonics 0..{N}   |   "
                  f"power captured = {100 * P_kept / P_total:.2f} %   |   "
                  f"MSE = {mse:.4f}   |   overshoot = {overshoot:.1f} % of jump")
    ax3.set_xlabel("Time (s)"); ax3.set_ylabel("Amplitude")
    ax3.set_ylim(-1.5 * A, 1.5 * A)
    ax3.legend(loc="upper right", ncol=6, fontsize=8)
    plt.show()

## 3. Interactive simulation

Use the controls:

| Control | Meaning |
|---|---|
| **N (harmonics)** | highest harmonic order kept in the reconstruction ($n = 0 \dots N$) |
| **Duty cycle D** | fraction of the period at $+A$ (0.5 = symmetric square wave) |
| **Delay τ** | time shift of the signal, in seconds (with $f_0 = 1$ Hz, $T_0 = 1$ s) |
| **Show harmonics** | draws each individual cosine that is being summed |

Start with **D = 0.5, τ = 0**, then move **N** from 1 upwards.

In [4]:
if HAS_WIDGETS:
    interact(
        plot_decomposition,
        N=widgets.IntSlider(value=5, min=1, max=99, step=1, description="N (harmonics)",
                            continuous_update=False, style={"description_width": "initial"},
                            layout=widgets.Layout(width="500px")),
        D=widgets.FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="Duty cycle D",
                              continuous_update=False, style={"description_width": "initial"}),
        tau=widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="Delay τ (s)",
                                continuous_update=False, style={"description_width": "initial"}),
        show_harmonics=widgets.Checkbox(value=False, description="Show harmonics"),
        A=widgets.fixed(1.0), f0=widgets.fixed(1.0), periods=widgets.fixed(2),
    );
else:
    plot_decomposition(N=5)

interactive(children=(IntSlider(value=5, continuous_update=False, description='N (harmonics)', layout=Layout(w…

## 4. Convergence study

How fast does the partial sum approach the square wave? The plot below shows the **fraction of signal power captured** (Parseval) and the **peak overshoot** as functions of $N$, for the symmetric case.

In [ ]:
Ns = np.arange(1, 200, 2)
t = np.linspace(0, 1, 20000, endpoint=False)
power, overshoot = [], []
for N in Ns:
    _, c = fourier_coefficients(N)
    amp, _ = one_sided_spectrum(c)
    power.append(100 * (amp[0] ** 2 + 0.5 * np.sum(amp[1:] ** 2)))
    overshoot.append((reconstruct(t, c).max() - 1) / 2 * 100)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
a1.plot(Ns, power, "o-", ms=3); a1.axhline(100, color="gray", ls="--")
a1.set_title("Signal power captured vs N"); a1.set_xlabel("N"); a1.set_ylabel("%")
a2.plot(Ns, overshoot, "o-", ms=3, color="tab:red")
a2.axhline(8.95, color="gray", ls="--", label="Gibbs limit ≈ 8.95 % of the jump")
a2.set_title("Peak overshoot vs N"); a2.set_xlabel("N"); a2.set_ylabel("% of jump 2A")
a2.legend(); plt.show()

print(f"With N = 1  (fundamental only): {power[0]:.2f} % of the power")
print(f"With N = 9:                      {power[4]:.2f} % of the power")

## 5. Check: analytic coefficients vs. FFT

Engineers rarely have a formula for the signal — they sample it and use the **FFT**. Here we sample exactly one period and compare the FFT‑based coefficients with the analytic ones.

*Notes:* small differences come from sampling the discontinuities. A phase of $+180^\circ$ and $-180^\circ$ is the same thing. A **negative DC value** appears as $A_0 = |c_0|$ with phase $180^\circ$.

In [ ]:
Ns_fft = 1024                       # samples in one period
D_test, tau_test = 0.3, 0.1         # try other values!
ts = np.arange(Ns_fft) / Ns_fft     # one period, f0 = 1 Hz
xs = square_wave(ts, D=D_test, tau=tau_test)
c_fft = np.fft.fft(xs) / Ns_fft     # c_n estimate = DFT / number of samples

n, c_an = fourier_coefficients(10, D=D_test, tau=tau_test)
amp_an, ph_an = one_sided_spectrum(c_an)
amp_fft, ph_fft = one_sided_spectrum(c_fft[:11], tol=1e-2)  # ignore tiny sampling leakage

print(" n |  A_n analytic   A_n FFT  |  phase analytic   phase FFT")
for k in range(11):
    print(f"{k:2d} |   {amp_an[k]:8.4f}   {amp_fft[k]:8.4f}  |   {ph_an[k]:8.1f}     {ph_fft[k]:8.1f}")

## 6. Exercises

1. **Odd harmonics only.** With $D = 0.5$, $\tau = 0$, which harmonics are present? Using the expression for $c_n$, prove that the even ones vanish. *(Hint: what is $e^{-j\pi n}$ for even $n$?)*
2. **Amplitude law.** Read $A_1$, $A_3$, $A_5$ from the plot and verify $A_n = 4A/(n\pi)$. How many dB per decade does the envelope fall?
3. **Gibbs phenomenon.** Increase $N$ up to 99. Does the overshoot disappear? What changes as $N$ grows? Compare with Section 4.
4. **Time shift.** Set $\tau = 0.25$ s (a quarter period). What happens to the amplitude spectrum? And to the phase spectrum? Express the phase change of harmonic $n$ as a function of $\tau$. Why does $\tau = 0.25$ s turn the sine series into a cosine series?
5. **Duty cycle.** Set $D = 0.25$. Which harmonics are now missing? Generalise: for $D = 1/k$, which harmonics vanish? What does the DC component $A_0$ become?
6. **Practical question.** A 1 kHz square wave from a function generator goes through a low‑pass filter with a 10 kHz cut‑off. Approximately how many harmonics pass? Use the simulator to sketch what the output looks like.
7. **Challenge.** Modify `square_wave` to create a **triangular wave** (or a sawtooth) and compute its spectrum with the FFT method of Section 5. Why do the triangle harmonics fall as $1/n^2$?